<a href="https://colab.research.google.com/github/aini43325018-ui/2025_algoritma_dan_pemrograman/blob/Pemrograman-Berbasis-Objek---Semester-2/Jobsheet_11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

01: Instalasi

In [ ]:
!pip install streamlit pandas
!npm install localtunnel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 47.9 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼
added 22 packages in 3s
⠴
⠴3 packages are looking for funding
⠴  run `npm fund` for details
⠴

02: konfigurasi.py

In [ ]:
%%writefile konfigurasi.py
import os

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
NAMA_DB = 'pengeluaran_harian.db'
DB_PATH = os.path.join(BASE_DIR, NAMA_DB)
KATEGORI_PENGELUARAN = ["Makanan", "Transportasi", "Hiburan", "Tagihan", "Belanja", "Kesehatan", "Pendidikan", "Lainnya"]
KATEGORI_DEFAULT = "Lainnya"

Writing konfigurasi.py


03: setup_db_pengeluaran.py

In [ ]:
%%writefile setup_db_pengeluaran.py
import sqlite3
import os
from konfigurasi import DB_PATH

def setup_database():
    print(f"Memeriksa/membuat database di: {DB_PATH}")
    conn = None
    try:
        conn = sqlite3.connect(DB_PATH)
        cursor = conn.cursor()
        sql_create_table = """
        CREATE TABLE IF NOT EXISTS transaksi (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            deskripsi TEXT NOT NULL,
            jumlah REAL NOT NULL CHECK(jumlah > 0),
            kategori TEXT,
            tanggal DATE NOT NULL
        );"""
        print(" Membuat tabel 'transaksi' (jika belum ada)...")
        cursor.execute(sql_create_table)
        conn.commit()
        print(" -> Tabel 'transaksi' siap.")
        return True
    except sqlite3.Error as e:
        print(f" -> Error SQLite saat setup: {e}")
        return False
    finally:
        if conn:
            conn.close()
            print(" -> Koneksi DB setup ditutup.")

if __name__ == "__main__":
    print("--- Memulai Setup Database Pengeluaran ---")
    if setup_database():
        print(f"\nSetup database '{os.path.basename(DB_PATH)}' selesai.")
    else:
        print(f"\nSetup database GAGAL.")
    print("--- Setup Database Selesai ---")

Writing setup_db_pengeluaran.py


04: database.py

In [ ]:
%%writefile database.py
import sqlite3
import pandas as pd
from konfigurasi import DB_PATH

def get_db_connection():
    try:
        conn = sqlite3.connect(DB_PATH, timeout=10, detect_types=sqlite3.PARSE_DECLTYPES)
        conn.row_factory = sqlite3.Row
        return conn
    except sqlite3.Error as e:
        print(f"ERROR [database.py] Koneksi DB gagal: {e}")
        return None

def execute_query(query: str, params: tuple = None):
    conn = get_db_connection()
    if not conn: return None
    try:
        cursor = conn.cursor()
        if params: cursor.execute(query, params)
        else: cursor.execute(query)
        conn.commit()
        return cursor.lastrowid
    except sqlite3.Error as e:
        print(f"ERROR [database.py] Query gagal: {e}")
        conn.rollback()
        return None
    finally:
        if conn: conn.close()

def fetch_query(query: str, params: tuple = None, fetch_all: bool = True):
    conn = get_db_connection()
    if not conn: return None
    try:
        cursor = conn.cursor()
        if params: cursor.execute(query, params)
        else: cursor.execute(query)
        return cursor.fetchall() if fetch_all else cursor.fetchone()
    except sqlite3.Error as e:
        print(f"ERROR [database.py] Fetch gagal: {e}")
        return None
    finally:
        if conn: conn.close()

def get_dataframe(query: str, params: tuple = None) -> pd.DataFrame:
    conn = get_db_connection()
    if not conn: return pd.DataFrame()
    try:
        df = pd.read_sql_query(query, conn, params=params)
        return df
    except Exception as e:
        print(f"ERROR [database.py] Gagal baca ke DataFrame: {e}")
        return pd.DataFrame()
    finally:
        if conn: conn.close()

def setup_database_initial():
    conn = get_db_connection()
    if not conn: return False
    try:
        cursor = conn.cursor()
        sql_create_table = """
        CREATE TABLE IF NOT EXISTS transaksi (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            deskripsi TEXT NOT NULL,
            jumlah REAL NOT NULL CHECK(jumlah > 0),
            kategori TEXT,
            tanggal DATE NOT NULL
        );"""
        cursor.execute(sql_create_table)
        conn.commit()
        return True
    except sqlite3.Error as e:
        print(f"Error SQLite saat setup tabel: {e}")
        return False
    finally:
        if conn: conn.close()

Writing database.py


05: model.py

In [ ]:
%%writefile model.py
import datetime

class Transaksi:
    """Merepresentasikan satu entitas transaksi pengeluaran (Data Class)."""
    def __init__(self, deskripsi: str, jumlah: float, kategori: str, tanggal: datetime.date | str, id_transaksi: int | None = None):
        self.id = id_transaksi
        self.deskripsi = str(deskripsi) if deskripsi else "Tanpa Deskripsi"
        try:
            jumlah_float = float(jumlah)
            self.jumlah = jumlah_float if jumlah_float > 0 else 0.0
        except (ValueError, TypeError):
            self.jumlah = 0.0
        self.kategori = str(kategori) if kategori else "Lainnya"

        if isinstance(tanggal, datetime.date):
            self.tanggal = tanggal
        elif isinstance(tanggal, str):
            try: self.tanggal = datetime.datetime.strptime(tanggal, "%Y-%m-%d").date()
            except ValueError: self.tanggal = datetime.date.today()
        else:
            self.tanggal = datetime.date.today()

    def to_dict(self) -> dict:
        return {
            "deskripsi": self.deskripsi,
            "jumlah": self.jumlah,
            "kategori": self.kategori,
            "tanggal": self.tanggal.strftime("%Y-%m-%d")
        }

Writing model.py


06: manajer_anggaran.py

In [ ]:
%%writefile manajer_anggaran.py
import datetime
import pandas as pd
from model import Transaksi
import database

class AnggaranHarian:
    _db_setup_done = False

    def __init__(self):
        if not AnggaranHarian._db_setup_done:
            if database.setup_database_initial():
                AnggaranHarian._db_setup_done = True

    def tambah_transaksi(self, transaksi: Transaksi) -> bool:
        if not isinstance(transaksi, Transaksi) or transaksi.jumlah <= 0: return False
        sql = "INSERT INTO transaksi (deskripsi, jumlah, kategori, tanggal) VALUES (?, ?, ?, ?)"
        params = (transaksi.deskripsi, transaksi.jumlah, transaksi.kategori, transaksi.tanggal.strftime("%Y-%m-%d"))
        last_id = database.execute_query(sql, params)
        if last_id is not None:
            transaksi.id = last_id
            return True
        return False

    def get_semua_transaksi_obj(self) -> list[Transaksi]:
        sql = "SELECT id, deskripsi, jumlah, kategori, tanggal FROM transaksi ORDER BY tanggal DESC, id DESC"
        rows = database.fetch_query(sql, fetch_all=True)
        transaksi_list = []
        if rows:
            for row in rows:
                transaksi_list.append(Transaksi(id_transaksi=row['id'], deskripsi=row['deskripsi'], jumlah=row['jumlah'], kategori=row['kategori'], tanggal=row['tanggal']))
        return transaksi_list

    def get_dataframe_transaksi(self, filter_tanggal: datetime.date | None = None) -> pd.DataFrame:
        # Ditambahkan kolom ID untuk memudahkan melihat transaksi yang akan dihapus
        query = "SELECT id, tanggal, kategori, deskripsi, jumlah FROM transaksi"
        params = None
        if filter_tanggal:
            query += " WHERE tanggal = ?"
            params = (filter_tanggal.strftime("%Y-%m-%d"),)
        query += " ORDER BY tanggal DESC, id DESC"
        df = database.get_dataframe(query, params=params)

        if not df.empty:
            try:
                import locale
                locale.setlocale(locale.LC_ALL, 'id_ID.UTF-8')
                df['Jumlah (Rp)'] = df['jumlah'].map(lambda x: locale.currency(x or 0, grouping=True, symbol='Rp ')[:-3])
            except:
                df['Jumlah (Rp)'] = df['jumlah'].map(lambda x: f"Rp {x or 0:,.0f}".replace(",", "."))
            # Menampilkan ID di DataFrame
            df = df[['id', 'tanggal', 'kategori', 'deskripsi', 'Jumlah (Rp)']]
        return df

    def hitung_total_pengeluaran(self, tanggal: datetime.date | None = None) -> float:
        sql = "SELECT SUM(jumlah) FROM transaksi"
        params = None
        if tanggal:
            sql += " WHERE tanggal = ?"
            params = (tanggal.strftime("%Y-%m-%d"),)
        result = database.fetch_query(sql, params=params, fetch_all=False)
        if result and result[0] is not None: return float(result[0])
        return 0.0

    def get_pengeluaran_per_kategori(self, tanggal: datetime.date | None = None) -> dict:
        hasil = {}
        sql = "SELECT kategori, SUM(jumlah) FROM transaksi"
        params = []
        if tanggal:
            sql += " WHERE tanggal = ?"
            params.append(tanggal.strftime("%Y-%m-%d"))
        sql += " GROUP BY kategori HAVING SUM(jumlah) > 0 ORDER BY SUM(jumlah) DESC"
        rows = database.fetch_query(sql, params=tuple(params) if params else None, fetch_all=True)
        if rows:
            for row in rows:
                kategori = row['kategori'] if row['kategori'] else "Lainnya"
                jumlah = float(row[1]) if row[1] is not None else 0.0
                hasil[kategori] = jumlah
        return hasil

    # Implementasi Logika Backend Fitur Hapus Transaksi
    def hapus_transaksi(self, id_transaksi: int) -> bool:
        sql = "DELETE FROM transaksi WHERE id = ?"
        try:
            database.execute_query(sql, (id_transaksi,))
            return True
        except Exception as e:
            print(f"Gagal menghapus transaksi: {e}")
            return False

Writing manajer_anggaran.py


07: main_app.py

In [ ]:
%%writefile main_app.py
import streamlit as st
import datetime
import pandas as pd
import locale

def format_rp(angka):
    try:
        locale.setlocale(locale.LC_ALL, 'id_ID.UTF-8')
        return locale.currency(angka or 0, grouping=True, symbol='Rp ')[:-3]
    except:
        return f"Rp {angka or 0:,.0f}".replace(",",".")

try:
    from model import Transaksi
    from manajer_anggaran import AnggaranHarian
    from konfigurasi import KATEGORI_PENGELUARAN
except ImportError as e:
    st.error(f"Gagal mengimpor modul: {e}. Pastikan file .py lain ada.")
    st.stop()

st.set_page_config(page_title="Catatan Pengeluaran", layout="wide")

@st.cache_resource
def get_anggaran_manager():
    return AnggaranHarian()

anggaran = get_anggaran_manager()

def halaman_input(anggaran: AnggaranHarian):
    st.header("💸 Tambah Pengeluaran Baru")
    with st.form("form_transaksi_baru", clear_on_submit=True):
        col1, col2 = st.columns([3, 1])
        with col1: deskripsi = st.text_input("Deskripsi", placeholder="Contoh: Makan siang")
        with col2: kategori = st.selectbox("Kategori", KATEGORI_PENGELUARAN, index=0)

        col3, col4 = st.columns([1, 1])
        with col3: jumlah = st.number_input("Jumlah (Rp)", min_value=0.01, step=1000.0, format="%.0f", value=None)
        with col4: tanggal = st.date_input("Tanggal", value=datetime.date.today())

        submitted = st.form_submit_button("💾 Simpan Transaksi")
        if submitted:
            if not deskripsi or jumlah is None or jumlah <= 0:
                st.warning("Deskripsi dan Jumlah wajib diisi dengan benar!", icon="⚠️")
            else:
                with st.spinner("Menyimpan..."):
                    tx = Transaksi(deskripsi, float(jumlah), kategori, tanggal)
                    if anggaran.tambah_transaksi(tx):
                        st.success(f"OK! Berhasil disimpan.", icon="✅")
                        st.cache_data.clear()
                        st.rerun()
                    else:
                        st.error("Gagal simpan.", icon="❌")

def halaman_riwayat(anggaran: AnggaranHarian):
    st.subheader("Detail Semua Transaksi")
    if st.button("🔄 Refresh Riwayat"):
        st.cache_data.clear()
        st.rerun()

    with st.spinner("Memuat riwayat..."):
        df_transaksi = anggaran.get_dataframe_transaksi()
        transaksi_list = anggaran.get_semua_transaksi_obj()

    if df_transaksi is None:
        st.error("Gagal ambil riwayat.")
    elif df_transaksi.empty:
        st.info("Belum ada transaksi.")
    else:
        # Menampilkan Data beserta ID-nya agar User tahu ID mana yang mau dihapus
        st.dataframe(df_transaksi, use_container_width=True, hide_index=True)

        st.divider()
        st.subheader("🗑️ Hapus Transaksi")

        # Ekstrak list ID untuk Dropdown
        id_tersedia = [t.id for t in transaksi_list] if transaksi_list else []

        if id_tersedia:
            id_hapus = st.selectbox("Pilih ID Transaksi yang ingin dihapus:", id_tersedia)
            if st.button("Hapus Transaksi Terpilih"):
                if anggaran.hapus_transaksi(id_hapus):
                    st.success(f"✅ Transaksi ID {id_hapus} berhasil dihapus!")
                    st.cache_data.clear()
                    st.rerun()
                else:
                    st.error("❌ Gagal menghapus transaksi.")

def halaman_ringkasan(anggaran: AnggaranHarian):
    st.subheader("Ringkasan Pengeluaran")
    col_filter1, col_filter2 = st.columns([1, 2])
    with col_filter1:
        pilihan_periode = st.selectbox("Filter Periode:", ["Semua Waktu", "Hari Ini", "Pilih Tanggal"])

    tanggal_filter = None
    label_periode = "(Semua Waktu)"
    if pilihan_periode == "Hari Ini":
        tanggal_filter = datetime.date.today()
        label_periode = f"({tanggal_filter.strftime('%d %b')})"
    elif pilihan_periode == "Pilih Tanggal":
        tanggal_filter = st.date_input("Pilih Tanggal:")
        label_periode = f"({tanggal_filter.strftime('%d %b %Y')})"

    with col_filter2:
        total_pengeluaran = anggaran.hitung_total_pengeluaran(tanggal=tanggal_filter)
        st.metric(label=f"Total Pengeluaran {label_periode}", value=format_rp(total_pengeluaran))

    st.divider()
    st.subheader(f"Pengeluaran per Kategori {label_periode}")
    dict_per_kategori = anggaran.get_pengeluaran_per_kategori(tanggal=tanggal_filter)

    if not dict_per_kategori:
        st.info(f"Tidak ada data untuk periode ini.")
    else:
        data_kategori = [{"Kategori": kat, "Total": jml} for kat, jml in dict_per_kategori.items()]
        df_kategori = pd.DataFrame(data_kategori).sort_values(by="Total", ascending=False).reset_index(drop=True)
        df_kategori['Total (Rp)'] = df_kategori['Total'].apply(format_rp)

        col_kat1, col_kat2 = st.columns(2)
        with col_kat1: st.dataframe(df_kategori[['Kategori', 'Total (Rp)']], hide_index=True, use_container_width=True)
        with col_kat2: st.bar_chart(df_kategori.set_index('Kategori')['Total'], use_container_width=True)

def main():
    st.sidebar.title("💰 Catatan Pengeluaran")
    menu_pilihan = st.sidebar.radio("Pilih Menu:", ["Tambah", "Riwayat", "Ringkasan"])
    st.sidebar.markdown("---")
    st.sidebar.info("Jobsheet - Aplikasi Keuangan")

    if menu_pilihan == "Tambah": halaman_input(anggaran)
    elif menu_pilihan == "Riwayat": halaman_riwayat(anggaran)
    elif menu_pilihan == "Ringkasan": halaman_ringkasan(anggaran)
    st.markdown("---")
    st.caption("Pengembangan Aplikasi Berbasis OOP")

if __name__ == "__main__":
    main()

Overwriting main_app.py


08: Inisialisasi Database

In [ ]:
!python setup_db_pengeluaran.py

--- Memulai Setup Database Pengeluaran ---
Memeriksa/membuat database di: /content/pengeluaran_harian.db
 Membuat tabel 'transaksi' (jika belum ada)...
 -> Tabel 'transaksi' siap.
 -> Koneksi DB setup ditutup.

Setup database 'pengeluaran_harian.db' selesai.
--- Setup Database Selesai ---


09: Menjalankan Streamlit & Localtunnel

In [ ]:
!wget -q -O - ipv4.icanhazip.com
!streamlit run main_app.py &>/content/logs.txt & npx localtunnel --port 8501

35.252.200.206
⠙⠹your url is: https://twelve-carpets-send.loca.lt
